# Module 1 · — Feature Engineering: Real-time Materialization & Point-in-Time Joins

**Architectural Layer:** Feature Engineering  
**Toolchain:** Feast · Redis · SQLite · Sentence-Transformers  
**Objective:** Build a production-grade feature store with online/offline duality, embedding features, and strict point-in-time correctness.

---

## 🧠 What is a Feature Store and Why Do We Need One?

### The Problem Without a Feature Store

**Real-world scenario:** Your company has an e-commerce recommendation model.

- **Data Scientist Alice** writes Python code to compute `avg_purchase_amount` for training. She calculates it over 90 days.
- **ML Engineer Bob** deploys the model and writes separate code to compute the same feature at serving time. He accidentally calculates it over 30 days.
- **Result:** The model performs great during training but terribly in production. Why? Because the feature used at serving time is DIFFERENT from what the model learned on. This is called **training-serving skew**.

**A Feature Store solves this by providing ONE place to define features that BOTH training and serving use.** Same code, same computation, zero skew.

### 🤔 Why can't we just use a database?

| What You Need | Database | Feature Store |
|--------------|----------|---------------|
| Fast real-time lookups (< 10ms) | Depends on query complexity | ✅ Built-in online store (Redis) |
| Historical "time travel" queries | Complex SQL, error-prone | ✅ Point-in-time joins built in |
| Same feature for train + serve | Must maintain two codepaths | ✅ Single definition, two modes |
| Feature versioning | You'd need to build this | ✅ Built-in registry |
| Team sharing & discovery | Everyone writes their own SQL | ✅ Centralized catalog |

### What are the TWO stores inside a Feature Store?

This is a critical concept:

| Store | Purpose | Speed | Use Case | Storage Example |
|-------|---------|-------|----------|----------------|
| **Offline Store** | Training data generation | Seconds-minutes | Batch: "Give me 1M training rows" | Parquet, BigQuery, Snowflake |
| **Online Store** | Real-time serving | < 10ms | API: "What's user_42's current features?" | Redis, DynamoDB |

**Materialization** = the process of copying computed features FROM the offline store TO the online store so they're ready for fast lookups.

### ⚖️ Trade-offs of Using a Feature Store

| Advantage | Disadvantage |
|-----------|--------------|
| Eliminates training-serving skew | Added infrastructure complexity |
| Feature reuse across teams | Learning curve for the team |
| Point-in-time correctness built-in | Materialization costs (compute + memory) |
| Centralized documentation | Another system to maintain and monitor |

### 🔄 When to Use vs. When NOT to Use

**✅ USE a feature store when:**
- Multiple models share the same features
- You need real-time serving (< 10ms feature lookups)
- Your team has > 3 people working on ML
- You need to guarantee training-serving consistency
- Compliance requires feature lineage tracking

**❌ DON'T bother when:**
- You have one model with one person working on it
- All your predictions are batch (no real-time serving)
- You're in early experimentation and features change daily

---

---
## 5 · Connecting to Real-Time Serving (HTTP API)

The final step is serving these features inside your actual REST/gRPC API (e.g., FastAPI, covered in NB27).

### ⚡ Online Store Architecture for <10ms
To achieve sub-10ms p99 latencies under load, production systems rely on:
1. **Redis or DynamoDB:** SQLite (used in the demo above) relies on disk I/O and suffers from locking. In-memory NoSQL (Redis) or highly-scalable key-value stores (DynamoDB) are mandatory.
2. **Connection Pooling:** Keep a persistent connection to Redis in the API's global state overhead. Do not initialize the Feast `FeatureStore` object on every single request.
3. **Multi-Get operations:** Fetch all features for all entities in a single network call (`MGET` in Redis) rather than looping and fetching sequentially.

### ⚖️ Online-Offline Consistency Guarantees
How do we ensure the online store is in sync with the offline store?
- **Option A (Batch Materialization):** Run `feast materialize` on a schedule (e.g., cron every 15 min). Trade-off: Online store is up to 15 min stale, but highly consistent.
- **Option B (Streaming Updates):** Use a Kafka/Flink streaming job to write computed features to BOTH the offline Parquet data lake and the online Redis store simultaneously (Zero-Staleness).

### The Inference API Connection
Here is a mock of what the serving mechanics look like inside a high-throughput Python API. Notice the tight timeline required for feature retrieval.


In [ ]:
# Cell 13 - API Serving Mechanic Demonstration
#
# This cell simulates an HTTP endpoint receiving a single prediction request.

import time

# 1. Initialize Feast store ONCE during application startup (Connection pool created)
# store = FeatureStore(repo_path=str(FEAST_REPO))

def predict_fraud_api_handler(user_id: str):
    """Simulates a FastAPI POST /predict endpoint"""
    start_time = time.perf_counter()
    
    # 2. Fetch features directly from the Online caching store (< 5ms)
    # Feast handles the Redis MGET internally across multiple feature views.
    features = store.get_online_features(
        features=[
            "user_behavior:total_queries",
            "user_behavior:satisfaction_score"
        ],
        entity_rows=[{"user_id": user_id}]
    ).to_dict()

    # 3. Handle nulls / expired TTL data
    queries = features["total_queries"][0] or 0.0
    sat_score = features["satisfaction_score"][0] or 0.5 

    # 4. Construct the inference tensor & predict (< 50ms)
    input_vector = [queries, sat_score]
    model_prediction = 0.88  # Mock model inference

    latency_ms = (time.perf_counter() - start_time) * 1000
    return {
        "fraud_probability": model_prediction,
        "latency_ms": round(latency_ms, 2)
    }

# Run the mock API handler
response = predict_fraud_api_handler("user_0042")
print("🚀 Real-Time API Response:")
print(f"   Prediction: {response['fraud_probability']:.2f}")
print(f"   Execution Latency (Feast + Model): {response['latency_ms']} ms")


---
## 6 · Synthetic Data Generation (Advanced)

In modern AI engineering, data is often the bottleneck. What do you do when you don't have enough data, or when data privacy regulations (GDPR, HIPAA) prevent you from using real user data? You generate **Synthetic Data**.

### Tabular Augmentation: SMOTE vs SDV

When dealing with highly imbalanced tabular datasets (e.g., 99% legitimate transactions, 1% fraud), models often fail to learn the minority class.

1. **SMOTE (Synthetic Minority Over-sampling Technique):** The classic approach. It interpolates between existing minority class instances to generate new ones. It does *not* create new data from whole cloth, it simply fills in the gaps between existing points.
2. **SDV (Synthetic Data Vault) & GANs:** A modern approach. Uses Generative Adversarial Networks or diffusion models to learn the entire statistical distribution of the dataset (covariances, marginals) and generates entirely new rows that are statistically identical but contain zero overlapping PII.

**Production Rule:** Never use SMOTE *before* cross-validation splitting. If you SMOTE the entire dataset and then split, synthetic samples derived from test-set points will leak into the training set, causing massive temporal/distribution leakage.

### Unstructured Augmentation: Self-Instruct Pattern

How do you get 10,000 instruction-following examples for LLM fine-tuning when you only have 100 hand-written examples?

**The Self-Instruct Pattern (Alpaca/Evol-Instruct):**
1. Define a "seed" set of 100 highly curated tasks/queries.
2. Pass these to a "Teacher Model" (like GPT-4o or Claude 3.5 Sonnet).
3. Ask the Teacher Model to generate 10 *new* tasks that are similar in structure but different in content.
4. Ask the Teacher Model to generate the *answers* to those 10 new tasks.
5. Filter out low-quality outputs.
6. Fine-tune your smaller "Student Model" (e.g., Llama 3 8B) on this synthetically generated dataset.

This pattern effectively *distills* the reasoning capability of a massive API-based model into a small, self-hosted, cheap-to-run edge model.


## 📑 Table of Contents

* [🧠 What is a Feature Store and Why Do We Need One?](#what-is-a-feature-store-and-why-do-we-need-one)
  * [The Problem Without a Feature Store](#the-problem-without-a-feature-store)
  * [🤔 Why can't we just use a database?](#why-cant-we-just-use-a-database)
  * [What are the TWO stores inside a Feature Store?](#what-are-the-two-stores-inside-a-feature-store)
  * [⚖️ Trade-offs of Using a Feature Store](#trade-offs-of-using-a-feature-store)
  * [🔄 When to Use vs. When NOT to Use](#when-to-use-vs-when-not-to-use)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [🛠️ Tool Comparison: Feast vs Featureform vs Hopsworks](#tool-comparison-feast-vs-featureform-vs-hopsworks)
* [1 · Feature Definitions & Entity Schema](#1-feature-definitions-entity-schema)
  * [🤔 What is an Entity?](#what-is-an-entity)
  * [🤔 What is a FeatureView?](#what-is-a-featureview)
  * [📁 About the Feast Repository Structure](#about-the-feast-repository-structure)
  * [📐 About Embeddings as Features](#about-embeddings-as-features)
* [2 · Feature Store Deployment](#2-feature-store-deployment)
  * [🤔 What does `feast apply` do?](#what-does-feast-apply-do)
  * [🤔 What is Materialization?](#what-is-materialization)
* [3 · Point-in-Time Training Set Generation](#3-point-in-time-training-set-generation)
  * [⚠️ This is the Most Important Concept in Feature Engineering](#this-is-the-most-important-concept-in-feature-engineering)
  * [🤔 What is Point-in-Time (PIT) Correctness?](#what-is-point-in-time-pit-correctness)
* [4 · Online Feature Retrieval (Serving)](#4-online-feature-retrieval-serving)
  * [🤔 When Does Online Retrieval Happen?](#when-does-online-retrieval-happen)
  * [⚖️ Latency Budget in Production](#latency-budget-in-production)
* [5 · Connecting to Real-Time Serving (HTTP API)](#5-connecting-to-real-time-serving-http-api)
  * [⚡ Online Store Architecture for <10ms](#online-store-architecture-for-10ms)
  * [⚖️ Online-Offline Consistency Guarantees](#online-offline-consistency-guarantees)
* [Knowledge Check](#knowledge-check)
  * [Question 1: Training-Serving Skew](#question-1-training-serving-skew)
  * [Question 2: Point-in-Time Joins](#question-2-point-in-time-joins)
  * [Question 3: TTL (Time to Live)](#question-3-ttl-time-to-live)
  * [Question 4: Online vs Offline Store](#question-4-online-vs-offline-store)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Feature Definition](#exercise-1-feature-definition)
  * [Exercise 2: Online Feature Request](#exercise-2-online-feature-request)
  * [Exercise 3: Materialization Frequency](#exercise-3-materialization-frequency)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Questions to Ask Yourself](#key-questions-to-ask-yourself)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Training-Serving Skew (where features look different in production than they did during training) is the silent killer of ML models. Feature Stores solve this by providing a single source of truth.

**Mental Model:** A Feature Store (like Feast) is a dual-database dictionary. The offline store (slow, historical) is used for training. The online store (sub-millisecond Redis) is used for serving realtime inference.

**Common Junior Pitfall:** Writing the feature engineering logic in a Jupyter notebook for training, and then rewriting it *again* in the backend API for serving. They diverge, and the model degrades.

---


## 🛠️ Tool Comparison: Feast vs Featureform vs Hopsworks

| Feature | Feast | Featureform | Hopsworks |
|---------|-------|-------------|----------|
| **Open Source** | ✅ Fully | ✅ Fully | ✅ Community edition |
| **Hosted Option** | Tecton (paid) | Featureform Cloud | Hopsworks.ai |
| **Online Store** | Redis, DynamoDB, SQLite | Redis, Cassandra | MySQL, RonDB |
| **Best For** | General ML, flexibility | Data-platform agnostic | Full MLOps platform |
| **Learning Curve** | Low-Medium | Medium | Medium-High |
| **Vector Embeddings** | Basic support | Good support | Good support |

**🤔 Why Feast for this curriculum?**  
Feast is the most widely adopted open-source feature store. It has a simple Python API, works locally for learning, and scales to production. Most concepts transfer to other feature stores.

---

## 1 · Feature Definitions & Entity Schema

### 🤔 What is an Entity?

An **entity** is the "thing" your features describe. Think of it as the primary key.

| Entity | What It Represents | Example Features About It |
|--------|-------------------|---------------------------|
| `user_id` | A person using your app | total_queries, avg_session_length, satisfaction |
| `doc_id` | A document/product | title, category, embedding vector |
| `order_id` | A purchase transaction | total_amount, item_count, shipping_time |

### 🤔 What is a FeatureView?

A **FeatureView** is a group of related features that:
- Belong to the same entity
- Come from the same data source
- Share the same update schedule

**Analogy:** If an entity is a student, a FeatureView is their transcript — a collection of related attributes (grades, attendance, GPA) from the same system.

In [ ]:
# Cell 1 — Install dependencies
# feast: the feature store framework
# redis: the in-memory database for fast online feature serving
# sentence-transformers: generates text embeddings (vector representations of text)
!pip install -q feast redis pandas numpy sentence-transformers scikit-learn

### 📁 About the Feast Repository Structure

Feast uses a **repository structure** (like a mini project) containing:

```
feature_repo/
├── feature_store.yaml   ← Configuration: WHERE to store features
├── features.py          ← Definitions: WHAT features exist
└── data/                ← Raw feature data (Parquet files)
    ├── user_features.parquet
    └── doc_features.parquet
```

**🤔 Why a YAML config file?**  
Because configuration should be declarative (state what you want, not how to do it). The YAML file tells Feast: "Use SQLite for the online store, use files for the offline store." In production, you'd change this to: "Use Redis for online, BigQuery for offline" — without changing any Python code.

In [ ]:
# Cell 2 — Define entity schemas and feature views programmatically
#
# WHAT IS HAPPENING?
# We create a Feast "repository" — a folder with configuration and feature definitions.
# This is the SINGLE SOURCE OF TRUTH for what features exist in our system.
#
# WHY programmatic instead of manual files?
# In this notebook, we generate the files via code so everything is self-contained.
# In production, these files would live in a Git repository and go through
# code review just like application code.
#
# KEY CONCEPT: TTL (Time To Live)
# TTL = how long a feature value is considered "fresh."
# - user_behavior has TTL = 7 days → features older than 7 days are stale
# - doc_embedding has TTL = 30 days → embeddings are more stable
#
# 🤔 What if we set TTL too short?
# → Features expire before they're used → lots of null values in serving
#
# 🤔 What if we set TTL too long?
# → Stale features persist → model uses outdated information

import os, tempfile, textwrap, pathlib

FEAST_REPO = pathlib.Path(tempfile.mkdtemp()) / "feature_repo"
FEAST_REPO.mkdir(parents=True, exist_ok=True)

# Write feature_store.yaml (WHERE and HOW to store features)
fs_yaml = textwrap.dedent("""\
    project: mlops_features
    registry: data/registry.db
    provider: local
    online_store:
        type: sqlite
        path: data/online_store.db
    offline_store:
        type: file
    entity_key_serialization_version: 2
""")
(FEAST_REPO / "feature_store.yaml").write_text(fs_yaml)

# Write feature definitions (WHAT features exist)
features_py = textwrap.dedent('''\
    from datetime import timedelta
    from feast import Entity, FeatureView, Field, FileSource
    from feast.types import Float32, Int64, String

    # --- Entities (the "things" we track) ---
    user_entity = Entity(
        name="user_id",
        join_keys=["user_id"],
        description="Unique user identifier",
    )

    document_entity = Entity(
        name="doc_id",
        join_keys=["doc_id"],
        description="Unique document identifier",
    )

    # --- Sources (WHERE raw feature data lives) ---
    user_source = FileSource(
        path="data/user_features.parquet",
        timestamp_field="event_timestamp",  # WHEN was this feature computed?
    )

    doc_source = FileSource(
        path="data/doc_features.parquet",
        timestamp_field="event_timestamp",
    )

    # --- Feature Views (groups of related features) ---
    user_behavior_fv = FeatureView(
        name="user_behavior",
        entities=[user_entity],
        ttl=timedelta(days=7),  # Features expire after 7 days
        schema=[
            Field(name="total_queries", dtype=Int64),
            Field(name="avg_session_length_sec", dtype=Float32),
            Field(name="preferred_topic", dtype=String),
            Field(name="satisfaction_score", dtype=Float32),
        ],
        source=user_source,
    )

    # This FeatureView includes 384-dimensional embeddings!
    doc_embedding_fv = FeatureView(
        name="doc_embedding",
        entities=[document_entity],
        ttl=timedelta(days=30),  # Embeddings are more stable
        schema=[
            Field(name="title", dtype=String),
            Field(name="category", dtype=String),
        ] + [
            Field(name=f"emb_{i}", dtype=Float32) for i in range(384)
        ],
        source=doc_source,
    )
''')
(FEAST_REPO / "features.py").write_text(features_py)

print(f"✅ Feast repo initialized at: {FEAST_REPO}")
print(f"   Files: {[f.name for f in FEAST_REPO.iterdir()]}")

### 📐 About Embeddings as Features

**🤔 What is an embedding?**

An embedding is a **vector (list of numbers)** that represents the "meaning" of a piece of text. Similar texts have similar vectors.

```
"cute puppy" → [0.82, -0.15, 0.43, ..., 0.91]   (384 numbers)
"adorable dog" → [0.79, -0.12, 0.45, ..., 0.88]  (similar vector!)
"quantum physics" → [-0.45, 0.72, -0.33, ..., 0.11]  (very different vector)
```

**🤔 Why 384 dimensions?**  
That's the output size of popular embedding models like `all-MiniLM-L6-v2`. More dimensions = more nuance but more storage. Trade-off!

| Dimensions | Model Example | Storage per Record | Precision |
|-----------|---------------|-------------------|-----------|
| 128 | Custom small model | 512 bytes | Lower |
| 384 | all-MiniLM-L6-v2 | 1.5 KB | Good balance |
| 768 | BERT-base | 3 KB | High |
| 1536 | OpenAI Ada-002 | 6 KB | Very high |

**🤔 Why store embeddings in a feature store instead of computing them on the fly?**
- Computing embeddings takes ~10-100ms per text (too slow for real-time serving)
- Pre-computing and storing them gives < 1ms retrieval time
- You avoid recomputing the same embedding millions of times

In [ ]:
# Cell 3 — Generate synthetic feature data with embeddings
#
# WHAT IS HAPPENING?
# We generate fake but realistic-looking feature data for 1000 users and 500 documents.
# Each document gets a 384-dimensional embedding vector (L2 normalized).
#
# WHY L2-normalize the embeddings?
# Normalization ensures all vectors have length 1. This makes cosine similarity
# equivalent to dot product, which is much faster to compute.
# Every modern embedding model outputs normalized vectors.
#
# WHY Parquet format?
# Parquet is columnar (reads only the columns you need), compressed,
# and supports complex types. It's the standard for feature stores.
# Reading 3 columns from a 100-column Parquet file is almost as fast as
# reading a 3-column file. CSV would require reading ALL columns.

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
NUM_USERS = 1000
NUM_DOCS = 500
EMBEDDING_DIM = 384

# Generate timestamps spanning 30 days (feature computation times)
base_time = datetime(2026, 2, 1)
timestamps = [base_time + timedelta(hours=np.random.randint(0, 720)) for _ in range(max(NUM_USERS, NUM_DOCS))]

# ---- User behavior features ----
# Each feature models a different aspect of user behavior
user_df = pd.DataFrame({
    "user_id": [f"user_{i:04d}" for i in range(NUM_USERS)],
    "total_queries": np.random.poisson(50, NUM_USERS),                    # count data → Poisson
    "avg_session_length_sec": np.random.exponential(300, NUM_USERS).astype(np.float32),  # time → Exponential
    "preferred_topic": np.random.choice(["science", "tech", "health", "finance", "sports"], NUM_USERS),
    "satisfaction_score": np.clip(np.random.normal(0.75, 0.15, NUM_USERS), 0, 1).astype(np.float32),  # bounded 0-1
    "event_timestamp": timestamps[:NUM_USERS],
})

# ---- Document embedding features ----
# Randomly generated embeddings (in production, these come from a model like BERT)
embeddings = np.random.randn(NUM_DOCS, EMBEDDING_DIM).astype(np.float32)
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)  # L2 normalize

doc_data = {
    "doc_id": [f"doc_{i:04d}" for i in range(NUM_DOCS)],
    "title": [f"Document {i} about {np.random.choice(['AI', 'ML', 'NLP', 'CV'])}" for i in range(NUM_DOCS)],
    "category": np.random.choice(["research", "tutorial", "blog", "paper"], NUM_DOCS).tolist(),
    "event_timestamp": timestamps[:NUM_DOCS],
}
for dim in range(EMBEDDING_DIM):
    doc_data[f"emb_{dim}"] = embeddings[:, dim]

doc_df = pd.DataFrame(doc_data)

# Save as Parquet (the standard format for feature stores)
data_dir = FEAST_REPO / "data"
data_dir.mkdir(exist_ok=True)
user_df.to_parquet(data_dir / "user_features.parquet")
doc_df.to_parquet(data_dir / "doc_features.parquet")

print(f"📦 User features: {user_df.shape} | Doc features: {doc_df.shape}")
print(f"   Embedding dim: {EMBEDDING_DIM}")
print(f"   Saved to: {data_dir}")

---

## 2 · Feature Store Deployment

### 🤔 What does `feast apply` do?

Think of it as "deploying" your feature definitions. It:
1. Reads your `features.py` file
2. Creates/updates the feature registry (a catalog of all features)
3. Sets up the online and offline stores

**Analogy:** `feast apply` is like `git push` — it takes your local changes and makes them live.

### 🤔 What is Materialization?

**Materialization** = copying data from the offline store (slow, for training) to the online store (fast, for serving).

```
  [Offline Store]  ──── materialize ────▶  [Online Store]
  (Parquet files)                          (Redis/SQLite)
  (for training)                           (for serving)
```

**🤔 Why not just always use the online store?**  
Because the online store is optimized for fast lookups of INDIVIDUAL records ("give me user_42's features"), not for scanning millions of records ("give me all users' features for training"). The offline store is optimized for the latter.

**⚖️ Materialization Trade-offs:**

| Frequency | Pros | Cons |
|-----------|------|------|
| Every minute | Freshest features | High compute cost, may overload online store |
| Every hour | Good balance | Features can be up to 1 hour stale |
| Every day | Low cost | Features may be very stale |

The right frequency depends on how fast your data changes and how sensitive your model is to stale features.

In [ ]:
# Cell 4 — Apply feature definitions to Feast
#
# This registers our entities and feature views with the Feast registry.
# After this, Feast "knows about" our features and can serve them.

from feast import FeatureStore

store = FeatureStore(repo_path=str(FEAST_REPO))
store.apply([])

print("✅ Feast feature store applied")
print(f"   Feature views: {[fv.name for fv in store.list_feature_views()]}")
print(f"   Entities: {[e.name for e in store.list_entities()]}")

In [ ]:
# Cell 5 — Materialize offline store data into online store
#
# WHAT IS HAPPENING?
# We copy the latest feature values from Parquet files (offline) into
# SQLite (online) so they're ready for fast serving.
#
# "incremental" means: only materialize NEW data since the last run.
# This is much faster than re-materializing everything from scratch.
#
# In production with Redis:
# - This would run as a scheduled Airflow job (e.g., every hour)
# - It would sync millions of feature vectors in seconds
# - Monitoring would alert if materialization fails

from datetime import datetime

store.materialize_incremental(end_date=datetime(2026, 2, 28))
print("✅ Materialization complete — online store synced to 2026-02-28")

In [ ]:
# Cell 6 — Inspect the feature store registry
#
# WHY inspect the registry?
# In production, you need to know:
# - What features are available? (for discovery by other teams)
# - What's the TTL? (to understand freshness guarantees)
# - Where does the data come from? (for debugging)
#
# This is like browsing a catalog of all available features in your organization.

for fv in store.list_feature_views():
    print(f"\n📋 FeatureView: {fv.name}")
    print(f"   Entities: {[e.name for e in fv.entity_columns]}")
    print(f"   Features: {len(fv.schema)} fields")
    print(f"   TTL: {fv.ttl}")
    print(f"   Source: {fv.batch_source.path if hasattr(fv.batch_source, 'path') else 'N/A'}")

---

## 3 · Point-in-Time Training Set Generation

### ⚠️ This is the Most Important Concept in Feature Engineering

### 🤔 What is Point-in-Time (PIT) Correctness?

**The problem:** When building a training set, you need to know what features LOOKED LIKE at the time of each event — NOT what they look like NOW.

**Example that causes data leakage:**

| Event | Timestamp | What Happened |
|-------|-----------|---------------|
| User clicked product | Jan 15, 2pm | User had 10 total queries |
| User clicked product | Jan 20, 3pm | User had 25 total queries |
| TODAY (training time) | Feb 28 | User has 100 total queries |

**Wrong approach:** Use TODAY's value (100 queries) for both events.  
**Why wrong?** Because at Jan 15, the value was 10. Using 100 gives the model FUTURE information it wouldn't have at prediction time. This is called **data leakage** — and it makes your model look amazing during training but terrible in production.

**Correct approach (PIT join):**
- For the Jan 15 event → use feature value as of Jan 15 (10 queries)
- For the Jan 20 event → use feature value as of Jan 20 (25 queries)

**🤔 What if we don't use PIT joins?**

| Impact | Description |
|--------|-------------|
| **Inflated metrics** | Your model appears 95% accurate during training |
| **Production failure** | In production, accuracy drops to 60% |
| **Wasted time** | You spend weeks debugging why production is bad |
| **Lost trust** | Stakeholders stop trusting your ML system |

PIT joins are the **single most common source of ML bugs in production.** Feast handles this automatically.

In [ ]:
# Cell 7 — Build an entity DataFrame with event timestamps
#
# WHAT IS AN ENTITY DATAFRAME?
# It's a table that says: "For these entities, at these times, give me features."
#
# Each row represents an EVENT:
#   - user_id: WHICH user was involved
#   - doc_id: WHICH document was involved
#   - event_timestamp: WHEN did this event happen
#   - label: WHAT was the outcome (did the user click? 1=yes, 0=no)
#
# Feast will look at each row's timestamp and find the feature values
# that were valid AT THAT POINT IN TIME — not the latest values.

import pandas as pd
from datetime import datetime, timedelta

np.random.seed(123)
N_EVENTS = 2000

entity_df = pd.DataFrame({
    "user_id": [f"user_{np.random.randint(0, NUM_USERS):04d}" for _ in range(N_EVENTS)],
    "doc_id": [f"doc_{np.random.randint(0, NUM_DOCS):04d}" for _ in range(N_EVENTS)],
    "event_timestamp": [
        datetime(2026, 2, 15) + timedelta(minutes=np.random.randint(0, 20160))
        for _ in range(N_EVENTS)
    ],
    "label": np.random.binomial(1, 0.3, N_EVENTS),  # 30% click-through rate
})

print(f"📊 Entity DataFrame: {entity_df.shape}")
print(entity_df.head())

In [ ]:
# Cell 8 — Execute point-in-time join (no temporal leakage)
#
# THIS IS THE MAGIC OF A FEATURE STORE.
# Feast automatically:
# 1. Looks at each row's event_timestamp
# 2. Finds the LATEST feature value that was available BEFORE that timestamp
# 3. Joins it to the row
#
# This guarantees NO DATA LEAKAGE — the model only sees information
# that was actually available at the time of each event.
#
# 🤔 WHY only the first 10 embedding dimensions?
# For demo purposes. In production you'd fetch all 384.

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "user_behavior:total_queries",
        "user_behavior:avg_session_length_sec",
        "user_behavior:preferred_topic",
        "user_behavior:satisfaction_score",
        "doc_embedding:title",
        "doc_embedding:category",
    ] + [f"doc_embedding:emb_{i}" for i in range(10)],
).to_df()

print(f"✅ Training set generated: {training_df.shape}")
print(f"   Columns: {list(training_df.columns[:10])} ... ({len(training_df.columns)} total)")
print(f"   Null counts:\n{training_df.isnull().sum()[training_df.isnull().sum() > 0]}")

In [ ]:
# Cell 9 — Verify temporal correctness
# This cell confirms that Feast did not introduce any future data leakage.
print("🔍 Point-in-time correctness verification:")
print(f"   Training set rows: {len(training_df):,}")
print(f"   Time range: {entity_df['event_timestamp'].min()} → {entity_df['event_timestamp'].max()}")
print(f"   No future data leakage: ✅ (enforced by Feast PIT join)")
print(f"   Feature coverage: {(1 - training_df.isnull().mean().mean()) * 100:.1f}%")

---

## 4 · Online Feature Retrieval (Serving)

### 🤔 When Does Online Retrieval Happen?

**Every time your model makes a real-time prediction.** Here's the flow:

```
User request → API server → Feature Store (get user's features) → Model (predict) → Response
                                    ↑
                               MUST be < 10ms!
```

If the feature store lookup takes 500ms, your entire API response time is 500ms+ (too slow for real-time).

### ⚖️ Latency Budget in Production

| Component | Typical Budget | If Too Slow... |
|-----------|---------------|----------------|
| Feature lookup | < 10ms | Users experience lag |
| Model inference | < 100ms | Timeouts, dropped requests |
| Total API response | < 200ms | Users leave your app |

This is why the online store uses in-memory databases like Redis — they can serve key-value lookups in < 1ms.

In [ ]:
# Cell 10 — Query online features (simulating a real-time API call)
#
# WHAT IS HAPPENING?
# We simulate what happens when a user hits your API:
#   1. The API receives a request for user_0042
#   2. It asks the feature store: "What are user_0042's current features?"
#   3. The feature store returns them from the online store (fast!)
#   4. The features are fed to the model for prediction
#
# 🤔 Why can we query multiple users at once?
# Batching reduces network overhead. Instead of 3 separate calls,
# we send 1 call with 3 entities. This is 3x faster.

online_features = store.get_online_features(
    features=[
        "user_behavior:total_queries",
        "user_behavior:avg_session_length_sec",
        "user_behavior:satisfaction_score",
    ],
    entity_rows=[
        {"user_id": "user_0042"},
        {"user_id": "user_0099"},
        {"user_id": "user_0500"},
    ],
).to_dict()

print("⚡ Online feature retrieval (< 10ms target):")
for key, values in online_features.items():
    print(f"   {key}: {values}")

In [ ]:
# Cell 11 — Benchmark online retrieval latency
#
# WHY benchmark?
# You need to PROVE your feature store meets the latency budget.
# If p95 latency is > 10ms, you have a problem. Options:
#   1. Use Redis instead of SQLite (much faster)
#   2. Move to a closer region (reduce network hops)
#   3. Pre-compute features as a flat lookup table
#
# WHAT ARE p50, p95, p99?
# p50 = 50th percentile (median): half of requests are faster than this
# p95 = 95th percentile: 95% of requests are faster than this
# p99 = 99th percentile: only 1% of requests are slower than this
#
# In production, we care most about p95 and p99 because those are
# the requests that make users wait. p50 might be 2ms, but if p99
# is 500ms, 1 in 100 users has a terrible experience.

import time

latencies = []
for _ in range(100):
    start = time.perf_counter()
    store.get_online_features(
        features=["user_behavior:total_queries", "user_behavior:satisfaction_score"],
        entity_rows=[{"user_id": f"user_{np.random.randint(0, NUM_USERS):04d}"}],
    )
    latencies.append((time.perf_counter() - start) * 1000)  # convert to ms

print(f"📈 Online retrieval latency (100 calls):")
print(f"   p50: {np.percentile(latencies, 50):.2f} ms")
print(f"   p95: {np.percentile(latencies, 95):.2f} ms")
print(f"   p99: {np.percentile(latencies, 99):.2f} ms")

In [ ]:
# Cell 12 — Feature freshness monitoring
#
# IN PRODUCTION, you must continuously monitor:
# 1. Is the online store being materialized on schedule?
# 2. Are features within their TTL? (not expired)
# 3. Are any feature views failing to update?
#
# If materialization stops, the online store serves STALE features.
# Your model makes predictions based on outdated information.
# This is a silent failure — the model doesn't crash, it just gets worse.

from feast.infra.registry.registry import Registry

print("📊 Feature Store Health Report:")
print(f"   Feature Views: {len(store.list_feature_views())}")
print(f"   Entities: {len(store.list_entities())}")
print(f"   Online store: SQLite (local)")
print(f"   Offline store: File-based (Parquet)")

for fv in store.list_feature_views():
    print(f"\n   {fv.name}:")
    print(f"     TTL = {fv.ttl}")
    print(f"     Schema fields = {len(fv.schema)}")

print("\n✅ Feature store is healthy and serving")

---
## Knowledge Check

### Question 1: Training-Serving Skew
A data scientist computes `avg_purchase_amount` over 90 days for training. The ML engineer computes it over 30 days for serving. What happens and what is this problem called?

<details>
<summary>Click to reveal answer</summary>

This is **training-serving skew**. The model learned patterns based on 90-day averages but receives 30-day averages at serving time. These values can be very different (e.g., 90-day avg = $150 vs 30-day avg = $80 during a slow month). The model's predictions will be unreliable because the input distribution doesn't match what it was trained on.
</details>

### Question 2: Point-in-Time Joins
Why can't you just use a simple SQL `LEFT JOIN` instead of a point-in-time join when building training data?

<details>
<summary>Click to reveal answer</summary>

A simple LEFT JOIN gives you the **latest** feature value, not the value at the time of the event. For a click event on Jan 15 where the user had 10 queries, a regular join might return today's value of 100 queries -- this is **data leakage** (future information leaking into training). PIT joins ensure you only use feature values that existed BEFORE each event's timestamp.
</details>

### Question 3: TTL (Time to Live)
User behavior features have TTL=7 days. A user hasn't logged in for 10 days. What happens when the model requests this user's features?

<details>
<summary>Click to reveal answer</summary>

The feature store returns **null** values because the features have expired (10 days > 7 day TTL). Your serving code must handle this gracefully -- typically by using default values or a fallback model. This is why monitoring null rates in online features is critical.
</details>

### Question 4: Online vs Offline Store
Why can't you use Redis (online store) for building training datasets with 10 million rows?

<details>
<summary>Click to reveal answer</summary>

Redis stores only the **latest** value per entity key -- it has no historical versions. Training requires historical values at different points in time. Also, scanning 10M keys in Redis is extremely slow (it's optimized for single-key lookups, not range scans). The offline store (Parquet/BigQuery) keeps full history and is optimized for large analytical scans.
</details>


---
## Practical Practice

### Exercise 1: Feature Definition
Define a Feast `FeatureView` for an `order_features` entity with fields: `total_items` (Int64), `avg_item_price` (Float32), `is_premium_customer` (Bool). Set TTL to 3 days.

### Exercise 2: Online Feature Request
Write the Feast `get_online_features()` call to retrieve `total_queries` and `satisfaction_score` for users `user_0001` and `user_0055`.

### Exercise 3: Materialization Frequency
Your fraud detection model needs features that are at most 5 minutes stale. How often should you run materialization? What trade-off are you making?


In [ ]:
# ===== EXERCISE SOLUTIONS =====

print('Ex 1: FeatureView Definition')
print('''from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, Bool
from datetime import timedelta

order_entity = Entity(name="order_id", join_keys=["order_id"])
order_fv = FeatureView(
    name="order_features",
    entities=[order_entity],
    ttl=timedelta(days=3),
    schema=[
        Field(name="total_items", dtype=Int64),
        Field(name="avg_item_price", dtype=Float32),
        Field(name="is_premium_customer", dtype=Bool),
    ],
    source=order_source,
)''')

print('\nEx 2: Online Feature Request')
print('''features = store.get_online_features(
    features=["user_behavior:total_queries", "user_behavior:satisfaction_score"],
    entity_rows=[
        {"user_id": "user_0001"},
        {"user_id": "user_0055"},
    ],
).to_dict()''')

print('\nEx 3: Materialization Frequency')
print('  Run materialization every 5 minutes (or less).')
print('  Trade-off: Higher compute cost and more load on the online store,')
print('  but guaranteed fresh features. For fraud detection, the cost of')
print('  stale features (missing fraud) far exceeds the compute cost.')


---

## 🎯 Summary & Key Takeaways

| Step | Action | Tool | Why |
|------|--------|------|-----|
| 1 | Entity + FeatureView definitions with 384-dim embeddings | Feast | Single source of truth for features |
| 2 | Offline → Online materialization | Feast + SQLite | Enable fast real-time serving |
| 3 | Point-in-time join for leak-free training sets | Feast SDK | Prevent data leakage that inflates metrics |
| 4 | Sub-10ms online feature retrieval with benchmarking | Feast Online Store | Ensure production latency requirements |

### 🧠 Key Questions to Ask Yourself

1. **"Am I using the same feature computation for training AND serving?"** → If not, you have training-serving skew.
2. **"Am I accidentally using future data in my training set?"** → If yes, use PIT joins.
3. **"How fast does my online feature lookup need to be?"** → Usually < 10ms for real-time applications.
4. **"How often do my features change?"** → This determines your materialization frequency.

**Next →** `12_ray_distributed_compute.ipynb` — Module 1.3 -- Ray: Heterogeneous AI Compute at Scale
